In [ ]:
%%sql -r dataframe_2
USE DATABASE ECOMMERCE;
USE SCHEMA ECOMMERCE_DN_SCHEMA;

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE DYNAMIC TABLE DT_REVENUE_GROWTH_TRACKER
TARGET_LAG = '1 hour'
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    order_date_key,
    source_channel,
    billing_currency,
    
    -- Core Financial Metrics
    SUM(line_item_subtotal) AS total_gross_revenue,
    SUM(discount_applied_amount) AS total_discounts_granted,
    SUM(line_item_subtotal - discount_applied_amount) AS total_net_revenue,
    
    -- Transaction Count for Average Order Value (AOV)
    COUNT(DISTINCT order_id) AS total_orders
FROM Sales_Invoices
GROUP BY 
    order_date_key, 
    source_channel, 
    billing_currency

In [ ]:
%%sql -r dataframe_5
SELECT * FROM DT_REVENUE_GROWTH_TRACKER LIMIT 20;

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE DT_PRODUCT_VELOCITY
TARGET_LAG = '1 hour'
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    order_date_key,
    product_key,
    order_status,
    
    -- Volume Metrics
    SUM(quantity) AS total_units_sold,
    COUNT(order_item_id) AS total_line_items_processed,
    
    -- Min/Max pricing bounds captured in the period
    MIN(unit_price) AS minimum_selling_price,
    MAX(unit_price) AS maximum_selling_price
FROM Sales_Invoices
GROUP BY 
    order_date_key, 
    product_key, 
    order_status;

In [ ]:
%%sql -r dataframe_6
SELECT * FROM DT_PRODUCT_VELOCITY LIMIT 20;

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE DT_FINANCIAL_RISK_AUDIT
TARGET_LAG = '1 hour'
WAREHOUSE = COMPUTE_WH
AS
SELECT 
    invoice_date_key,
    payment_status,
    payment_method,
    is_disputed,
    
    -- Risk & Leakage Measures
    SUM(late_fee_charged) AS total_late_fees_levied,
    COUNT(CASE WHEN is_disputed = TRUE THEN 1 END) AS total_disputed_lines,
    SUM(CASE WHEN is_disputed = TRUE THEN (line_item_subtotal - discount_applied_amount) ELSE 0 END) AS net_revenue_at_risk,
    
    -- Total baseline to measure the percentage of overall risk
    SUM(line_item_subtotal - discount_applied_amount) AS baseline_net_revenue
FROM Sales_Invoices
GROUP BY 
    invoice_date_key, 
    payment_status, 
    payment_method, 
    is_disputed;

In [ ]:
%%sql -r dataframe_7
SELECT * FROM DT_FINANCIAL_RISK_AUDIT LIMIT 20;